In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

import matplotlib.pyplot as plt
import importlib
import numpy as np

from VariablesClass import VariablesClass
from StructureClass import  StructureClass
from StateClass import StateClass
from EquilibriumClass import EquilibriumClass
from SupervisorClass import SupervisorClass
from config import CFG

import plot_funcs, colors, helpers_builders, file_funcs, numerical_experiments

## Average training time

In [ ]:
folder = "Training\\Apr23randomPosAfterFroceExplode"
average_t = file_funcs.average_successful_train_time(folder = folder, thresh=1e-6) 
print("average_t = ", average_t)

## Non abelianity check

In [ ]:
import config
importlib.reload(config)
from config import CFG

# Non-abelian buckle check: inputs
importlib.reload(plot_funcs)
importlib.reload(numerical_experiments)

Strctr = StructureClass(CFG, update_scheme=CFG.Train.update_scheme)
Variabs = VariablesClass(Strctr, CFG)
Sprvsr = SupervisorClass(Strctr, CFG, supress_prints=False)
Sprvsr.create_dataset(Strctr, CFG, CFG.Train.dataset_sampling, tip_pos=None, tip_angle=None)

init_buckle = helpers_builders._initiate_buckle(
    CFG.Strctr.H,
    CFG.Strctr.S,
    buckle_pattern=CFG.Train.init_buckle_pattern,
)

flat_tip_pos = np.array([Strctr.edges * Strctr.L, 0.0])
flat_tip_angle = 0.0

# Edit these values for the check.
initial_tip_pos = np.array([0.87 * Strctr.edges * Strctr.L, -0.001])
initial_tip_angle = -0.02
final_tip_pos = np.array([0.5 * Strctr.edges * Strctr.L, -0.3 * Strctr.edges * Strctr.L])
final_tip_angle = -np.pi/2
Eq_iterations = 2

print("flat pose:", flat_tip_pos, flat_tip_angle)
print("initial pose:", initial_tip_pos, initial_tip_angle)
print("final pose:", final_tip_pos, final_tip_angle)
print("initial buckle:", np.asarray(init_buckle, dtype=int).reshape(-1))

In [ ]:
# Run the warm start and both operation orders.
non_abelian_result = numerical_experiments.check_non_abelianity(
    Strctr,
    Variabs,
    Sprvsr,
    CFG,
    init_buckle,
    flat_tip_pos,
    flat_tip_angle,
    initial_tip_pos,
    initial_tip_angle,
    final_tip_pos,
    final_tip_angle,
    Eq_iterations,
    verbose=True,
)

print("\nFinal buckle after position -> angle:", non_abelian_result["buckle_position_then_angle"])
print("Final buckle after angle -> position:", non_abelian_result["buckle_angle_then_position"])
print("Non-abelian in buckle space:", non_abelian_result["non_abelian"])

## Fast coverage of transitions

In [ ]:
# Hamming-transition coverage from exported training or tip-sweep files
from collections import Counter
from pathlib import Path
import re
import pandas as pd

importlib.reload(file_funcs)
importlib.reload(plot_funcs)
importlib.reload(helpers_builders)

N_BITS = 4
# folder = Path(r"Training\Apr23randomPosAfterFroceExplode")
folder = Path(r"grid sweep\separate files\tip_grid_transition_csvs")
# folder = Path(r"Training\May17Like_noFlipChain")
plots_folder = Path("efficient_transitions")
plots_folder.mkdir(exist_ok=True)
mod = "sweep"  # "training" selects tasks; "sweep" selects initial graph nodes
if mod not in {"training", "sweep"}:
    raise ValueError("mod must be 'training' or 'sweep'")
save_transition_frames = True
frames_folder = plots_folder / f"{mod}_html_frames"
frames_folder.mkdir(exist_ok=True)

transitions, per_file_transitions, per_file_loss, edge_zero_loss_count, missing_edges = file_funcs.buckle_transitions(
    folder=folder,
    only_init_and_final_buckles=False,
    omit_inverted=True,
    transition_mode="hamming",
    reciprocity=False,
)

required_hamming_transitions = {
    edge
    for edge in helpers_builders.all_possible_transitions(N_BITS)
    if helpers_builders.hamming_distance_int(*edge) == 1
}


def parse_training_task_bits(file_name: str) -> tuple[str, str]:
    """Extract initial and desired buckle strings from a training filename."""
    init_match = re.search(r"init_([01]+)", file_name)
    desired_match = re.search(r"desired_?([01]+)", file_name)
    if init_match is None or desired_match is None:
        raise ValueError(f"Could not parse init/desired buckles from {file_name}")
    return init_match.group(1), desired_match.group(1)


def parse_sweep_init_bits(file_name: str) -> str:
    """Extract the initial buckle from an init_*_finalTip sweep filename."""
    match = re.fullmatch(r"init_([01]+)_finalTip_x=.*_y=.*_theta=.*\.csv", file_name)
    if match is None:
        raise ValueError(f"Could not parse sweep initial buckle from {file_name}")
    return match.group(1)


def hamming_only(edges):
    """Keep only directed transitions between adjacent Hamming rows."""
    return [edge for edge in edges if helpers_builders.hamming_distance_int(*edge) == 1]


def save_transition_diagram(counter: Counter, missing_edges, path: Path, init_bits: str, desired_bits: str | None) -> None:
    """Save a transition diagram without displaying every intermediate figure inline."""
    original_show = plt.show
    plt.show = lambda *args, **kwargs: None
    try:
        plot_funcs.plot_transition_diagram(
            counter,
            transitions_between_runs=False,
            only_reached_nodes=False,
            edge_zero_loss_count=Counter(),
            missing_edges=missing_edges,
            layout="hamming",
            initial_state=init_bits,
            desired_state=desired_bits,
        )
        plt.gcf().savefig(path, dpi=200, bbox_inches="tight")
    finally:
        plt.close("all")
        plt.show = original_show


rng = np.random.default_rng(0)
state_visit_counts = Counter()
ordered_files = []
selected_node_order = []


def states_visited_in_run(init_bits, run_edges):
    visited = {init_bits}
    for src, dst in run_edges:
        visited.add(helpers_builders.index_to_buckle(src, N_BITS))
        visited.add(helpers_builders.index_to_buckle(dst, N_BITS))
    return visited


if mod == "training":
    available_tasks = {}
    for file_name, edges in sorted(per_file_transitions.items()):
        init_bits, desired_bits = parse_training_task_bits(file_name)
        task_key = (init_bits, desired_bits)
        if init_bits == desired_bits or task_key in available_tasks:
            continue
        available_tasks[task_key] = (file_name, init_bits, desired_bits, hamming_only(edges))

    def add_training_task(task_key):
        task = available_tasks.pop(task_key)
        ordered_files.append((*task, len(selected_node_order)))
        selected_node_order.append(task_key)
        state_visit_counts.update(states_visited_in_run(task[1], task[3]))

    first_task = ("0" * N_BITS, "1" * N_BITS)
    if first_task not in available_tasks:
        raise FileNotFoundError(f"Could not find {first_task} to use as training step 0.")
    add_training_task(first_task)

    while available_tasks:
        scores = {
            task_key: state_visit_counts[task_key[0]] + state_visit_counts[task_key[1]]
            for task_key in available_tasks
        }
        best_score = min(scores.values())
        candidates = [task_key for task_key, score in scores.items() if score == best_score]
        add_training_task(candidates[int(rng.integers(len(candidates)))])

else:
    available_sweeps = {}
    for file_name, edges in sorted(per_file_transitions.items()):
        init_bits = parse_sweep_init_bits(file_name)
        available_sweeps.setdefault(init_bits, []).append(
            (file_name, init_bits, None, hamming_only(edges))
        )

    def add_sweep_node(init_bits):
        node_files = available_sweeps.pop(init_bits)
        selection_index = len(selected_node_order)
        selected_node_order.append(init_bits)
        for sweep_file in node_files:
            ordered_files.append((*sweep_file, selection_index))
            state_visit_counts.update(states_visited_in_run(sweep_file[1], sweep_file[3]))

    first_node = "0" * N_BITS
    if first_node not in available_sweeps:
        raise FileNotFoundError(f"Could not find sweep files beginning with init_{first_node}_.")
    add_sweep_node(first_node)

    while available_sweeps:
        best_score = min(state_visit_counts[node] for node in available_sweeps)
        candidates = [node for node in available_sweeps if state_visit_counts[node] == best_score]
        add_sweep_node(candidates[int(rng.integers(len(candidates)))])

    expected_sweep_files = 2304
    if len(ordered_files) != expected_sweep_files:
        print(f"Warning: found {len(ordered_files)} sweep files; expected {expected_sweep_files}.")
    print(f"Sweep node order: {selected_node_order}")

cumulative_transitions = Counter()
coverage_rows = []
first_step = 1 if mod == "sweep" else 0

for file_index, (file_name, init_bits, desired_bits, run_edges, selection_index) in enumerate(ordered_files):
    step = file_index + first_step
    run_counter = Counter(run_edges)
    cumulative_transitions.update(run_counter)
    observed = set(cumulative_transitions)
    missing_hamming_transitions = sorted(required_hamming_transitions - observed)

    coverage_rows.append({
        "step": step,
        "mod": mod,
        "node_selection": selection_index,
        "file": file_name,
        "init_buckle": init_bits,
        "desired_buckle": desired_bits,
        "run_hamming_transition_events": int(sum(run_counter.values())),
        "run_unique_hamming_transitions": int(len(run_counter)),
        "cumulative_hamming_transition_events": int(sum(cumulative_transitions.values())),
        "cumulative_unique_hamming_transitions": int(len(observed)),
        "missing_hamming_transitions": int(len(missing_hamming_transitions)),
    })

    is_last_file_for_selection = (
        file_index == len(ordered_files) - 1
        or ordered_files[file_index + 1][4] != selection_index
    )
    if save_transition_frames and is_last_file_for_selection:
        png_path = frames_folder / f"{selection_index}_init_{init_bits}.png"
        save_transition_diagram(cumulative_transitions, missing_hamming_transitions, png_path, init_bits, desired_bits)

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv("cumulative_transition.csv", index=False)


In [ ]:
importlib.reload(plot_funcs)

plot_funcs.plot_cumulative_transition_curve(
    coverage_df,
    x_col="step",
    x_label="sweep" if mod == "sweep" else "training task",
    save_path=plots_folder / "cumulative_transition.png",
)
html_path = plot_funcs.make_png_slider_html(
    frames_folder,
    html_path=plots_folder / "transition_animation.html",
    glob_pattern="*_init_*.png",
    title="Transition animation",
)

observed_hamming_transitions = set(cumulative_transitions)
missing_hamming_transitions = sorted(required_hamming_transitions - observed_hamming_transitions)
print(
    f"Hamming coverage: {len(observed_hamming_transitions)}/{len(required_hamming_transitions)} "
    f"directed one-bit transitions; missing {len(missing_hamming_transitions)}."
)
print("Saved cumulative CSV to cumulative_transition.csv")
print(f"Saved transition PNGs to {frames_folder}")
print(f"Saved transition HTML to {html_path}")

# coverage_df